<a href="https://colab.research.google.com/github/ShadenAhmed/DataScience-Project/blob/main/DateScience_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1. Collecting Data


##1.1 Gold prices

In [11]:
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
!pip install gnews

In [12]:

gold_symbol = "GC=F"
start_date = "2020-01-01"
end_date = "2026-1-31"

gold_raw_data = yf.download(gold_symbol, start=start_date, end=end_date)

print(gold_raw_data.head()) # first five days of 2020
print(gold_raw_data.tail()) # last days in 2026

gold_raw_data.to_csv("gold_prices_raw.csv")


/tmp/ipykernel_1065/133012054.py:5: FutureWarning: YF.download() has changed argument auto_adjust default to True
  gold_raw_data = yf.download(gold_symbol, start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed


Price             Close         High          Low         Open Volume
Ticker             GC=F         GC=F         GC=F         GC=F   GC=F
Date                                                                 
2020-01-02  1524.500000  1528.699951  1518.000000  1518.099976    214
2020-01-03  1549.199951  1552.699951  1530.099976  1530.099976    107
2020-01-06  1566.199951  1580.000000  1560.400024  1580.000000    416
2020-01-07  1571.800049  1576.300049  1558.300049  1558.300049     47
2020-01-08  1557.400024  1604.199951  1552.300049  1579.699951    236
Price             Close         High          Low         Open  Volume
Ticker             GC=F         GC=F         GC=F         GC=F    GC=F
Date                                                                  
2026-01-26  5079.700195  5095.600098  5052.200195  5081.500000     180
2026-01-27  5079.899902  5079.899902  5079.899902  5079.899902      34
2026-01-28  5301.600098  5301.600098  5301.600098  5301.600098  112054
2026-01-29  53

## 1.2 Geopolitical News
Collected in phase 1 but the size of data were too small so in phase 2 we decide to collect more data from the same source to improve data quality for conducting analysis

In [ ]:

from gnews import GNews
import pandas as pd
import time

google_news = GNews(
    language='en',
    max_results=100,
    start_date=(2020,1,1),
    end_date=(2026,1,1)
)

keywords = [
"geopolitical tensions",
"war",
"geopolitical risk",
"global conflict",
"russia ukraine",
"middle east tensions",
"political instability",
"economic sanctions",
"financial crisis",
"inflation crisis"
]

articles = []

for word in keywords:
    news = google_news.get_news(word)

    for n in news:
        articles.append({
            "title": n["title"],
            "date": n["published date"],
            "url": n["url"],
            "topic": word
        })

    time.sleep(2)

df = pd.DataFrame(articles)

print("Total geopolitical news:", len(df))

df.to_csv("raw_news.csv", index=False)

## 1.3 Global geopolitical conflict

Our secondary dataset

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/ShadenAhmed/DataScience-Project/refs/heads/main/geopolitical_conflict_risk_dataset.csv"
df = pd.read_csv(url)

columns_needed = [
    "country",
    "month",
    "inflation_rate",
    "gdp_growth_pct",
    "border_disputes_count",
    "sanctions_active",
    "instability_score",
    "conflict_escalation_6m"
]

df = df[columns_needed]

df["month"] = pd.to_datetime(df["month"], errors="coerce")

df = df.dropna(subset=["month"])

df.head()

Uploaded the secondary dataset and chooses the relevant columns also convert the month column to datetime to ensure its consistent before perform EDA

#2. Data Processing and Cleaning


## 2.1 Geopolitical news data

In [ ]:
#imports + read

import pandas as pd
import re
import matplotlib.pyplot as plt

df = pd.read_csv("raw_news.csv")

#inspect dataset

df.head()
df.info()
df.isnull().sum()
df.describe()

**Inspection result:** date is stored as object type and requires conversion to datetime. Duplicate rows were found while no missing values were detected in the dataset.

In [ ]:
#cleaning the data

nltk.download('stopwords')
nltk.download('wordnet')

# Date conversion to datetime
df['date'] = pd.to_datetime(df['date'], format='mixed', utc=True).dt.date


#Remove duplicates by url to be more accurate
df = df.drop_duplicates(subset=["url"])

#Even though no missing titles were found during the previous step,
#we still need to ensure the dataset is clean and reliable before further analysis.
df = df.dropna(subset=["title"])

#cleening the text
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):

    text = re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)
    text = re.sub(r'\W', ' ', text)
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)

    # To lowercase
    words = text.lower().split()

    # Lemmatization
    clean_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]

    return " ".join(clean_words)

#side by side comparison
df["title_clean"] = df["title"].apply(clean_text)
df[["title", "title_clean"]].head(5)

df.head()

We will create a binary **event indicator** using specific keywords to identify geopolitical headlines, converting the news text into structured data that can be compared with gold price movements:

In [ ]:
#fearure engineering:
#convert unstucutured text to stucutured text by creating an event indicator

keywords = ["war", "conflict", "invasion", "attack",
            "sanction", "military", "tension", "crisis","crises"]

def detect_event(text):
    return 1 if any(word in text for word in keywords) else 0
df["event_indicator"] = df["title_clean"].apply(detect_event)
df["event_indicator"].value_counts()

#a monthgly event count
df["year_month"] = pd.to_datetime(df["date"]).dt.to_period("M")
monthly_events = df.groupby("year_month")["event_indicator"].sum()

monthly_events

News events will be organized by **month** to identify periods of higher geopolitical activity:

In [ ]:
#sort by date
df = df.sort_values("date")
df[["date", "title_clean", "event_indicator"]].head()

In [ ]:
#save clean dataset

clean_news_data = df[[
    "date",
    "title_clean",
    "event_indicator",
    "year_month"
]].copy()

clean_news_data.to_csv("clean_news_data.csv", index=False)

print("Clean dataset saved:", clean_news_data.shape)

##2.2 Gold prices data


In [ ]:
import pandas as pd

df = pd.read_csv("gold_prices_raw.csv")
#Initial Inspection
df.head()
df.info()
df.describe()

In [ ]:
#Renaming Columns
df.rename(columns={'Price': 'Date'}, inplace=True)
df.head()

In [ ]:
#Removing Metadata Rows
df = df.iloc[2:].reset_index(drop=True)
df.head()

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])
#Converting Data Types
numeric_cols = ['Close','High','Low','Open','Volume']
df[numeric_cols] = df[numeric_cols].astype(float)
df.head()

In [ ]:
#Removing Duplicates
df.duplicated().sum()
df.drop_duplicates(inplace=True)

In [ ]:
#Sorting by Date
df.sort_values('Date', inplace=True)
df.reset_index(drop=True, inplace=True)

In [ ]:
#Calculate daily returns
df['Daily_Return'] = df['Close'].pct_change()
df.head()

In [ ]:
#Compute daily volatility
df['Volatility'] = df['High'] - df['Low']
df.head()

In [ ]:
#Add new variable (Year) from the Date column
df['Year'] = df['Date'].dt.year
df.head()

In [ ]:
#Handling Missing Values
df.isnull().sum()
df.dropna(inplace=True)
df.head()

In [ ]:
#Outliers analyzing
Q1 = df['Daily_Return'].quantile(0.25)
Q3 = df['Daily_Return'].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df['Daily_Return'] < Q1 - 1.5*IQR) |
              (df['Daily_Return'] > Q3 + 1.5*IQR)]
print(outliers.head())

Since the outliers is important in analyze the effects of geopolitical events we decide to kept it.

In [ ]:
#Saved cleaned and processed dataset
df.to_csv("gold_prices_cleaned.csv", index=False)


#3. Exploratory Data Analysis

## 3.1 EDA for Gold prices primary data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#Determinig the styles and colors of the graphs
sns.set_theme(style="whitegrid")
vibrant_gold = "#FFD700"
sky_blue = "#1E90FF"
sunset_orange = "#FF4500"

# Loading the gold price data
df = pd.read_csv('gold_prices_cleaned.csv')
df['Date'] = pd.to_datetime(df['Date'])


#Checking how big our data is and what kind of information we have.
print("Checking the size of the dataset...")
print(f"Dataset Size: {df.shape[0]} rows and {df.shape[1]} columns")

# Looking at the data types (Advanced_eda style)
print("\nWhat kinds of data are in each column?")
print(pd.value_counts(df.dtypes))

# Visualizing how unique the data is
plt.figure(figsize=(10, 5))
unique_counts = df.nunique().sort_values()
unique_counts.plot.bar(color=sky_blue, title="Unique Values per Column (Log Scale)")
plt.yscale('log')
plt.ylabel("Unique Count")
plt.xlabel("Columns")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


#Checking for any missing information or messy data.
print("\nAre there any missing values in our dataset?")
print(df.isnull().sum())


#STATISTICAL SUMMARIES
print("\nSummary statistics for our gold price data:")
print(df.describe())

#TREND ANALYSIS (Visualizing Prices)
#Checking how gold prices changed over time
plt.figure(figsize=(14, 7))
plt.plot(df['Date'], df['Close'], color=vibrant_gold, linewidth=2, label='Closing Price')
plt.fill_between(df['Date'], df['Close'], color=vibrant_gold, alpha=0.1)
plt.title('Gold Price Trends: 2020 - 2026', fontsize=16, fontweight='bold')
plt.xlabel('Year', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend()
plt.show()

#PATTERN ANALYSIS (Distribution of Returns)
#This helps in determining the most common daily price changes
plt.figure(figsize=(12, 6))
sns.histplot(df['Daily_Return'], bins=60, kde=True, color=sunset_orange)
plt.title('Daily Returns Pattern (Normal Distribution?)', fontsize=16, fontweight='bold')
plt.xlabel('Daily Return Percentage')
plt.show()

#CORRELATION ANALYSIS (Feature Relationships)
plt.figure(figsize=(10, 8))
numeric_only = df.select_dtypes(include=[np.number])
correlation_matrix = numeric_only.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='YlGnBu', fmt='.2f', square=True)
plt.title('Correlation Heatmap: How features connect', fontsize=16, fontweight='bold')
plt.show()

#ANOMALY DETECTION
#Spotting price jumps (outliers)
plt.figure(figsize=(12, 6))
sns.boxplot(x='Year', y='Daily_Return', data=df, palette="husl")
plt.title('Yearly Daily Returns: Spotting Anomalies', fontsize=16, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Daily Return')
plt.show()

###Final EDA Summary (Gold Prices Primary Data)

1. Price Trends and Patterns

 Gold prices were steady for a few years, but starting in 2024, they took off, . Looking at the “daily returns” (how much it changes each day), most of the movements are small and centered around 0%, following a standard bell curve. This means the price growth is consistent rather than just random jumps.

2. Anomalies (The “Weird” Days)

By using boxplots for each year, we spotted some clear outliers. 2020 and 2022 had some crashes, where the price dropped much harder than usual. However, 2024 and 2025 looks a lot more stable even though the price is higher. This suggests the current bull market has less extreme panic than we saw a few years ago.

3. Key Relationship (Correlation)
The heatmap showed that while all the price points (Open, Close, High, Low) move perfectly together, Volume actually has almost no relationship with the price.This is an important insight because it tells us that a high number of trades does not necessarily mean the price will move in a certain direction.

At the end commenting on what was mentioned above, the data is high quality and shows a strong, trending market with occasional historical shocks.

## 3.2 EDA for Geopolitical news primary data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer

df_news = pd.read_csv("clean_news_data.csv")
# Convert date to datetime object
df_news['date'] = pd.to_datetime(df_news['date'])

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (14, 7)
custom_green = '#2F4A2A'

In [ ]:
#draw barplot of monthly geopolitical event count
monthly_events.plot(kind="bar", figsize=(10,5), color="red")
plt.title("Monthly Geopolitical Event Count")
plt.xlabel("Year-Month")
plt.ylabel("Number of Events")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

#high activity months
monthly_events.sort_values(ascending=False).head(5)

This graph shows the number of geopolitical events reported in the news each month. Based on the results, we observed that some months were more active than others, particularly June 2025, October 2024, and December 2025, indicating that these periods saw a rise in geopolitical tensions or conflicts.


In [ ]:
# News distribution per year
plt.figure(figsize=(8,5))

df_news.groupby(df_news.date.dt.year).size().plot(kind='bar')

plt.title("Number of Geopolitical News Articles per Year")
plt.xlabel("Year")
plt.ylabel("Number of Articles")
plt.xticks(rotation=0)
plt.tight_layout()

plt.show()

This bar chart shows the number of geopolitical news articles published each year between 2020 and 2026. We noticed that some years contain more news articles than others, especially the later years, and this shows that they contain more geopolitical events or conflicts.


In [ ]:
 #Top Bigrams
from sklearn.feature_extraction.text import CountVectorizer

def plot_trends(corpus, n=15):
    vectorizer = CountVectorizer(ngram_range=(2,2))
    X = vectorizer.fit_transform(corpus)
    sum_words = X.sum(axis=0)

    words_freq = [(word, sum_words[0, idx]) for word, idx in vectorizer.vocabulary_.items()]
    words_freq = sorted(words_freq, key=lambda x: x[1], reverse=True)
    trend_df = pd.DataFrame(words_freq[:n], columns=['Bigram','Frequency'])

    plt.figure(figsize=(8,6))
    sns.barplot(x='Frequency', y='Bigram', data=trend_df, palette='summer')

    plt.title(f'Top {n} Geopolitical & Economic events')
    plt.show()

plot_trends(df_news['title_clean'])

This chart shows the main geopolitical and economic topics that have been commonly covered in the news, such as (the Middle East, financial crises, geopolitical risk, and global conflicts).


In [ ]:
# Anomaly detection
# Daily news volume
daily_news = df_news.groupby('date').size()

# Statistical
mean_vol = daily_news.mean()
std_vol = daily_news.std()

# Detect anomalies
anomalies = daily_news[daily_news > (mean_vol + 2 * std_vol)]

# Plot
plt.figure(figsize=(12,6))

plt.plot(daily_news.index, daily_news.values, label='Daily Volume', color='gray', alpha=0.6)

plt.scatter(anomalies.index, anomalies.values,
            color='red', s=60,
            label='Anomalies',
            zorder=5)

plt.title('Major Geopolitical News Spikes')
plt.xlabel("Date")
plt.ylabel("Number of Articles")

plt.xticks(rotation=90)

plt.legend()
plt.tight_layout()
plt.show()

This graph shows the number of daily geopolitical news articles over time. The gray line represents the typical daily news volume and the red dots represent unusual spikes in news activity and these spikes occur when the number of articles is higher than the average and that indicates significant geopolitical events or crises that have received extensive media coverage

In [ ]:
#word cloud
from wordcloud import WordCloud

# Combine all headlines
all_text = " ".join(df_news['title_clean'])

WordCloud = WordCloud(width=1000,
               height=500,
               background_color='white',
               colormap='summer',
               max_words=150).generate(all_text)

# Plot
plt.figure(figsize=(12,6))
plt.imshow(WordCloud, interpolation='bilinear')
plt.axis('off')

plt.title('Word Cloud of Geopolitical News')
plt.show()

This chart provides a visual summary of the most frequently words in news headlines. Larger words indicate words that appear most often, reflecting their importance in the geopolitical topics covered by the news articles. These words include (war, Middle East, and global conflicts)

## 3.3 EDA of merged data (gold prices + Geopolitical news)

In [ ]:
df_daily_news = df_news.groupby('date').size().reset_index(name='news_volume')
df_combined = pd.merge(df, df_daily_news, left_on='Date', right_on='date', how='inner')
df_combined.head()

In [ ]:
# detect spikes
# Use 'news_volume' from df_combined as the event count for daily analysis
mean_events = df_combined["news_volume"].mean()
std_events = df_combined["news_volume"].std()

spikes = df_combined[df_combined["news_volume"] > mean_events + 2*std_events]

# plot
plt.figure(figsize=(10,5))

# Plot daily Close price for gold
plt.plot(df_combined['date'], df_combined["Close"], label="Gold Price" , color='gold')

plt.scatter(spikes['date'],
            spikes["Close"],
            label="Event spikes")

plt.xticks(rotation=90)
plt.title("Gold price during geopolitical news spikes")
plt.xlabel("Date")
plt.ylabel("Gold price")
plt.legend()

plt.tight_layout()
plt.show()

This visualization shows the daily gold price with markers indicating dates when the geopolitical news volume significantly spiked. This allows us to observe how gold prices behave during periods of high geopolitical activity.

In [ ]:
# correlation matrix
# Select only numeric columns relevant for correlation
numeric_cols = ['Close', 'news_volume']
corr_matrix = df_combined[numeric_cols].corr()

# heatmap
plt.figure(figsize=(6,4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')

plt.title("Correlation Between Gold Prices and Geopolitical Events")
plt.show()

The correlation analysis between gold prices and news volume showed a weak relationship indicating that news volume alone does not definitively explain price movements. However, one possible explanation is that financial markets may react to major events with a delay, meaning that the impact of geopolitical news on gold prices may appear a few days later rather than on the same day.

In [ ]:
df_combined['event_level'] = df_combined['news_volume'].apply(lambda x: "High" if x > df_combined['news_volume'].mean() else "Low")

sns.boxplot(x='event_level', y='Close', data=df_combined)

plt.title("Gold Price During High vs Low Geopolitical Activity")
plt.show()

This boxplot compares gold prices during months with high and low geopolitical activity. If the median price during high activity is higher it may indicate that gold prices tend to increase during periods of geopolitical tension. In this boxplot we observe that in month of high geopolitical activity gold prices increases

## 3.4 EDA for secondary data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
secondary_df = pd.read_csv('geopolitical_conflict_risk_dataset.csv')

In [ ]:
display(secondary_df.head())
secondary_df.info()

In [ ]:
display(secondary_df.describe())

We used the describe() function to extract statistical values, such as the mean, standard deviation, and maximum and minimum values. The goal was to determine the normal value of the instability score. We found that the mean was approximately 66, which suggests a normal value for us. We also observed an anomaly with the maximum value reaching 149.69. This significant difference indicates that geopolitical events have occurred that have impacted instability

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=secondary_df, x='instability_score', bins=30, kde=True)
plt.show()

We used a histogram to understand how the values ​​are distributed. The goal is to see where most of the data cluster (between 60 and 80) and to confirm that there is a rightward slope that statistically proves the presence of rare anomalies exceeding 140, which we identified in the statistical summary.

In [ ]:
secondary_df['month'] = pd.to_datetime(secondary_df['month'])
global_trend = secondary_df.groupby('month')['instability_score'].mean().reset_index()
plt.figure(figsize=(12, 6))
sns.lineplot(data=global_trend, x='month', y='instability_score', marker='o')
plt.xticks(rotation=45)
plt.show()

We used a line chart to visualize the evolution of the instability score from 2020 to 2025. The goal of graph is to track data over time and identify specific periods where there were spikes in geopolitical risks such as late 2023.

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(data=secondary_df, x='instability_score')
plt.show()

We used a boxplot to visualize the spread of data around the median and to accurately identify outliers. This helps us see values ​​that exceed the normal range of the instability index, confirming the unusual spikes (such as the value of 149.69) we observed in the statistical summary.

In [ ]:
corr_matrix = secondary_df.select_dtypes(include=['float64', 'int64']).corr()
plt.figure(figsize=(18, 14))
sns.heatmap(corr_matrix , cmap='coolwarm', center=0)
plt.show()

We used a heatmap to measure the strength of the relationship between the instability score and other measures. Our goal is to confirm that increased protests and conflicts raise this score, while decreased political stability leads to increased instability. This analysis helps us understand why the data spikes and changes at some periods.